# 📊 Data Analysis with Python

<p xmlns:cc="http://creativecommons.org/ns#" xmlns:dct="http://purl.org/dc/terms/">This <a property="dct:title" rel="cc:attributionURL" href="https://github.com/mauriciohbc/data-analysis-notebook">Data Analysis Notebook</a> by <a rel="cc:attributionURL dct:creator" property="cc:attributionName" href="https://www.linkedin.com/in/mhbcardoso/">Maurício Henrique BEZERRA CARDOSO</a> is licensed under <a href="https://creativecommons.org/licenses/by-nc-sa/4.0/?ref=chooser-v1" target="_blank" rel="license noopener noreferrer" style="display:inline-block;">Creative Commons Attribution-NonCommercial-ShareAlike 4.0 International<img src="https://mirrors.creativecommons.org/presskit/icons/cc.svg?ref=chooser-v1" alt="drawing" width="22"><img  src="https://mirrors.creativecommons.org/presskit/icons/by.svg?ref=chooser-v1" alt="drawing" width="22"><img  src="https://mirrors.creativecommons.org/presskit/icons/nc.svg?ref=chooser-v1" alt="drawing" width="22"><img src="https://mirrors.creativecommons.org/presskit/icons/sa.svg?ref=chooser-v1" alt="drawing" width="22"></a></p>

## 👋🏾 Welcome to this notebook for Data Analysis with Python

This notebook is divided in two parts.

In Part 1, you will explore a dataframe using the library [pandas](https://pandas.pydata.org/docs/index.html). With this library, you will do many manipulations in your data, such as data import, date parsing, dropping unwanted row and columns, create new columns, resample your data and make simple plots.

In Part 2, you will get to explore two new libraries: [plotly](https://plotly.com/python/), for data visualization and [statsforecast](https://nixtlaverse.nixtla.io/statsforecast/index.html), for forecasting. This notebook is not intended to deep dive into data visualization and timeseries forecasting, but you will get a nice opportunity to discover these two topics and make a step forward in your data journey.

## ⚠️ Be sure you read the instructions before moving on to writing some code

We have taken the time to explain you a lot of stuff to facilitate your task of writing your code. So please, despite all your enthusiasm in writing a code very fast and hit the play button, do read all instructions. You will know what method to use and understand what is going on.

## 📚 Requirements install and libraries import

Before starting any project, you must be sure you have all libraries required for your work.

You can do that by listing your libraries in a file called `requirements.txt` and then making a command `pip install -r requirements.txt`.

If you have many different projects sharing the same environment (your pc, for example), you might want to create virtual environments. We are not getting into details on this right now.

As you do not need too many libraries, let us install them directly with a `pip install`:

In [ ]:
!pip install pandas plotly statsforecast numpy

Now you can import the libraries you need and set some general parameters for the rest of the notebook.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA
pd.set_option("display.max_columns", None)

## 🐼 Part 1 - Data Analysis with pandas

In this notebook, you will use data on electricity production and consumption in France, made available by RTE (Réseau de Transport d'Electricité), the electricity transmission system operator of France. Raw data can be found in the [here](https://www.rte-france.com/eco2mix/telecharger-les-indicateurs). You will be given all details necessary to understand the data along this notebook. If you want to know more, you can find more information [here](https://assets.rte-france.com/prod/public/2020-07/%C3%A9CO2mix%20-%20Description%20des%20fichiers%20des%20donn%C3%A9es%20en%20puissance.pdf), in French.

To facilitate our work, you can find all raw data in this [github repo](https://github.com/mauriciohbc/rte-raw-data).

So, let us start having fun with the data 😃.

First, let us import the data from 2012. As the files are in a `.xls` format, we might think that `pd.read_excel` (documentation [here](https://pandas.pydata.org/docs/reference/api/pandas.read_excel.html)) would be a good call. Well... In fact, for this data it is not true. Let us try anyway. 

❗️Spoiler alert: it will not work.

In [ ]:
pd.read_excel(f"https://github.com/mauriciohbc/rte-raw-data/raw/refs/heads/main/eCO2mix_RTE_Annuel-Definitif_2012.xls")

Read the last phrase of the error. It says that the Excel file format cannot be determined. Indeed, if you open the file with a text editor, you will notice that the file looks more like a csv with a tab separator.

### Exercise 1️⃣ - Import data from 2012 using `pd.read_csv`

Use `pd.read_csv` (documentation [here](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html)) to import data from 2012. Please, notice that you will have to set some parameters to the right value:
* The `encoding` is latin-1, because columns name are in French
* The separator (`sep`) is tab (`\t`)
* `index_col` must be set to `False`, to force pandas not to take the first column as index
* `skipfooter` must be set to `2`, because in the end of every file there are two lines (one with a legal mention and another empty), which are not useful for you
* `engine` must be set to `python`, so you can use the parameters `skipfooter`

In [ ]:
pd.read_csv(f"https://github.com/mauriciohbc/rte-raw-data/raw/refs/heads/main/eCO2mix_RTE_Annuel-Definitif_2012.xls",
            encoding = "latin-1",
            sep = "\t",
            index_col=False,
            engine = "python",
            skipfooter = 2)

### Exercise 2️⃣ - Get data from 2012 to 2022 using a for loop

Now that you know how to import the data correctly, you can use a for loop to create a dictionary with all dataframes from 2012 to 2022. For this, you need to use f-strings. To get a sense on how f-strings work, check the following code:

In [ ]:
for number in range(1,6):
    print(f"This is the sentence number {number}")

Notice that all urls have the same format:

```python
f"https://github.com/mauriciohbc/rte-raw-data/raw/refs/heads/main/eCO2mix_RTE_Annuel-Definitif_{year}.xls"
```

Use this f-string a for loop to get a dictionary with the dataframes from 2012 to 2022.

In [ ]:
df_dict = {}
for year in range(2012, 2023):
    df_dict[year] = pd.read_csv(f"https://github.com/mauriciohbc/rte-raw-data/raw/refs/heads/main/eCO2mix_RTE_Annuel-Definitif_{year}.xls",
                                  encoding = "latin-1",
                                  sep = "\t",
                                  engine = "python",
                                  index_col=False,
                                  skipfooter = 2)

### Exercise 3️⃣ - Concatenante all dataframes using `pd.concat`

Now that you have imported all dataframes in a dictionary, you can concatenate all of them in a single dataframe using `pd.concat` (documentation [here](https://pandas.pydata.org/docs/reference/api/pandas.concat.html)).

In [ ]:
power_df = pd.concat(df_dict)

Let us have a look on what `power_df` looks like:

In [ ]:
power_df.head(n=10)

Before moving on, it is essential to understand the data. This is an important step for any data project. Be sure you do not neglect it.

The dataframe ```power_df``` gives us information on the consumption of electricity (column `Consommation`), with a 30 min frequency. It also gives us the forecast of the consumption made one day before and some minutes before (column `Prévion J-1` and `Prévision J`), given every 15 min. That is why there is no value for consumption for every 15 or 45 minutes past the hour, as frequency for consumption and prediction are not the same.

The other columns refer to the production in power by different sources: fuel oil, coal, gas, nuclear, wind, solar, hydraulic and bioenergies (columns `Fioul`, `Charbon`, `Gaz`, `Nucléaire`, `Eolien`, `Solaire`, `Hydraulique` and `Bioenergies`). The production is also impacted by power spent on hydraulic pumping (column `Pompage`) and the physical exchange to other broad countries (column `Ech. physiques`, positive or negative).

The other columns give for details for each source. For the physical exchanger of power, there are some details for some countries. There are also some more details for gas, hydraulics, bioenergy and wind.

Note that the units for all columns referring to power consumption and production is MW (Mega Watt).

### Exercise 4️⃣ - Get rid of unwanted rows

For the sake of this notebook, you can keep only the rows with information on prediction and consumption. That is to say that all rows with a NaN value in the column `Consommation` can be dropped. For this, you can use `pd.DataFrame.dropna` (documentation [here](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.dropna.html)). Pay close attention to the parameters `axis`, `how` and `subset`.

It is a good idea to make a copy of the dataframe to test if you are doing the dropping correctly. You can make a copy a dataframe with using `pd.DataFrame.copy` (documentation [here])

In [ ]:
power_df_copy = power_df.copy()
power_df_copy.dropna(subset=["Consommation"])

After the dropping the unwanted rows, there must be 192 864 rows: (11 years * 365 days/years + 3 days for leap years)*(24 hours / day) * (2 samples / hour). If it is not the case, you are probably not dropping the rows correctly, so please revise your code. That is why we have made a copy of the original dataframe. 

If you are confident your dropping is good, you can apply it to your original dataframe, using the parameter `inplace` set to True (read more about it in the documentation [here](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.dropna.html)).

In [ ]:
power_df.dropna(subset=["Consommation"], inplace = True)

### Exercise 5️⃣ - Keep only the wanted columns

For the sake of this notebook, let us keep only the columns concerning the date, hour, consumption, forecast and production by each macro source. Concretly, let us keep only the following columns: `Date`,	`Heures`, `Consommation`, `Prévision J-1`, `Prévision J`, `Fioul`, `Charbon`, `Gaz`, `Nucléaire`, `Eolien`, `Solaire`, `Hydraulique`, `Pompage`, `Bioénergies`, `Ech. physiques`.

Let us take advatange of this manipulation and rename the columns in English: `Date`, `Hour`, `Consumption`, `Forecast D-1`, `Forecast D`, `Fuel oil`, `Coal`, `Gas`, `Nuclear`, `Wind`, `Solar`, `Hydraulic`, `hydraulic pumping`, `Bioenergies` and `Physical exchange`.

To keep only the wanted columns, you can use `pd.DataFrame.loc` (documentation [here](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.loc.html)). To rename the columns, you can change the attribute `pd.DataFrame.columns` (documentation [here](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.columns.html)). Finally, we can also get rid of the multilevel index by resetting it, using `pd.DataFrame.reset_index`(documentation [here](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.reset_index.html)). Pay attention to the `drop` parameters when resetting the index.

In [ ]:
columns_in_French = ["Date", "Heures", "Consommation", "Prévision J-1", "Prévision J", "Fioul", "Charbon", "Gaz", "Nucléaire", "Eolien", "Solaire", 
                     "Hydraulique", "Pompage", "Bioénergies", "Ech. physiques"]

columns_in_English = ["Date", "Hour", "Consumption", "Forecast D-1", "Forecast D", "Fuel oil", "Coal", "Gas", "Nuclear", "Wind", "Solar", 
                      "Hydraulic", "Hydraulic pumping", "Bioenergies", "Physical exchange"]

simple_power_df = power_df.loc[:, columns_in_French]
simple_power_df.columns = columns_in_English 
simple_power_df = simple_power_df.reset_index(drop=True)

Let us check `simple_power_df`:

In [ ]:
simple_power_df

### Exercise 6️⃣ - Choose the right dtype for each column

Let us use `pd.DataFrame.info` (documentation [here](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.info.html)) to have important info on `simple_power_df`:

In [ ]:
simple_power_df.info()

The documentation of our data tells us that all columns except `Date` and `Hours` should be integer. You did not end up with integers in most of the columns because of the NaN values you initially had. You can set the columns to the right dtype using `pd.DataFrame.astype` (documentation [here](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.astype.html)):

In [ ]:
dtype_dict = dict.fromkeys(columns_in_English, "int64")
dtype_dict["Date"] = "object"
dtype_dict["Hour"] = "object"

simple_power_df = simple_power_df.astype(dtype_dict)  

You can also work the date to have the full information (date and hour) in the same column. For this, we will use `pd.to_datetime`(documentation [here](https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html)).

In [ ]:
simple_power_df["ds"] = pd.to_datetime(simple_power_df["Date"] + ' ' + simple_power_df["Hour"])

Check if the column `ds` looks ok:

In [ ]:
simple_power_df["ds"]

If it does look ok, then you can drop the columns `Date` and `Hour` using `pd.DataFrame.drop` (documetation [here](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.drop.html)):

In [ ]:
simple_power_df = simple_power_df.drop(columns=["Date", "Hour"])

You can now check `simple_power_df` with `pd.DataFrame.info` to check the dtypes:

In [ ]:
simple_power_df.info()

### Exercise 7️⃣ - Create new columns

An important classification for electricity sources is  if they are low-carbon or high-carbon. You can create new columns with the sum of the pertinent columns. You can use `pd.DataFrame.sum` (documentation [here](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sum.html)). Pay attention to the `axis` you must choose. For info, you can consider:

* Low-carbon: `Nuclear`, `Wind`, `Solar`, `Hydraulic`, `Hydraulic pumping` (as it lows the electricity available by Hydraulic) and `Bioenergies`
* High-carbon: `Fuel oil`, `Coal`, `Gas`

Another interesting classification is the availability of the source. Some sources can be available 24 hours a day, such as nuclear, but others scources are intermittent, such as wind and solar. You can use the following classification:

* High-availability: `Fuel oil`, `Coal`, `Gas`, `Nuclear`, `Hydraulic`, `Hydraulic pumping`(as it lows the electricity available by Hydraulic) and `Bioenergies`
* Intermittent: `Wind`, `Solar`

For now, we do not have information on how to classify the `Physical exchange`, so you can disconsider it for both classifications.

In [ ]:
simple_power_df["Low-carbon"] = simple_power_df[["Nuclear", "Wind", "Solar", "Hydraulic", "Hydraulic pumping", "Bioenergies"]].sum(axis="columns")
simple_power_df["High-carbon"] = simple_power_df[["Fuel oil", "Coal", "Gas"]].sum(axis="columns")
simple_power_df["High-availability"] = simple_power_df[["Fuel oil", "Coal", "Gas", "Nuclear", "Hydraulic", "Hydraulic pumping", "Bioenergies"]].sum(axis="columns")
simple_power_df["Intermittent"] = simple_power_df[["Wind", "Solar"]].sum(axis="columns")

### Exercise 8️⃣ - Resample the dataframe

It can be convenient for some usecases to have the data in high period, instead of each 30 min, you can have at daily, monthly or even yearly. You can do that using `pd.DataFrame.resample` (documentation [here](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.replace.html)).

For this transformation, you need to be careful. Summing power in this case can be misleading. You'd better sum energy. Considering that the power is constant over each 30-min interval, you can use the following formula:

$$Energy = Power \cdot \Delta t$$

Remember, in this case, $\Delta t = 0.5$, because our interval between samples is 30 min (0,5 hour). The unit becomes MWh. For monthly and yearly data, you can divide the result by 1e6 to have data in TWh.

In [ ]:
daily_energy_df = simple_power_df.resample("D", on="ds").sum()*0.5
monthly_energy_df = daily_energy_df.resample("ME").sum()
yearly_energy_df = monthly_energy_df.resample("YE").sum()

### Transform MWh into TWh

monthly_energy_df = monthly_energy_df/1e6
yearly_energy_df = yearly_energy_df/1e6

### Exercise 9️⃣ - Plot the data

Let us check `monthly_energy_df`:

In [ ]:
monthly_energy_df

You can notice that `ds` became index. Let's get a visual from this data using `pd.DataFrame.plot` (documentation [here](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.plot.html)):

In [ ]:
monthly_energy_df.plot()

All in all, not the best plot. The legend can be brought outside adding the following code after the `plot` method:
```python
.legend(loc='center left',bbox_to_anchor=(1.0, 0.5))
```
You can try to change the figure size using the parameter `figsize`.
Try again to see if you get a better plot:

In [ ]:
monthly_energy_df.plot(figsize=(20,10)).legend(loc='center left',bbox_to_anchor=(1.0, 0.5))

As the variables have different intervals, you may want to separate them in groups of smilar range. Check the `subplot` parameter. As a suggestion, you can group the following variables in the same plot:

* `Consumption`, `Forecast D-1`, `Forecast D`, `Nuclear`
* `Gas`, `Wind`, `Hydraulic`
* `Bioenergies`, `Solar`, `Fuel oil`
* `High-carbon`, `Low-carbon`

You may want to adapt the size of your figure.

In [ ]:
monthly_energy_df.plot(figsize=(15,20),
                      subplots=[
                          ('Consumption', 'Forecast D-1', 'Forecast D', 'Nuclear'),
                          ('Gas', 'Wind', 'Hydraulic'),
                          ('Bioenergies', 'Solar', 'Fuel oil'),
                          ('High-carbon', 'Low-carbon')
                      ])

There are some improvements for those data visualization. In part 2, you will see a library for data visualization. But hey, great work so far this first part. You were able to:

* Import different dataframes with a for-loop
* Concatenate all data into a single dataframe
* Get rid of unwanted rows
* Keep only certain columns
* Choose the right dtype for each column
* Create new columns
* Resample the dataframe for higher periods and
* Get some plot to visualize the data.

All this with a single library: pandas 🐼

Now you can deep dive into the second part of this notebook, in which you will discover two new libraries, useful for your data journey in Python.

## 🔍 Part 2 - Exploring other libraries

For this second part, you need the dataframes created in Part 1. If you haven't done Part 1 or are only interested in Part 2, you can import the dataframes in the following code:

In [ ]:
monthly_energy_df = pd.read_csv("https://raw.githubusercontent.com/mauriciohbc/data-analysis-notebook/refs/heads/master/monthly_energy_production_and_consumption_in_France.csv",
                                index_col="ds", parse_dates=["ds"])
yearly_energy_df = pd.read_csv("https://raw.githubusercontent.com/mauriciohbc/data-analysis-notebook/refs/heads/master/yearly_energy_production_and_consumption_in_France.csv",
                               index_col="ds", parse_dates=["ds"])

### 📈 Data Visualization with Plotly

[Plotly](https://plotly.com/python/getting-started/) is a library for data visualization which produces interactive charts with very few lines of code.

There are two ways to use plotly: plotly express or graph objects. The former allows you to produce chart with very few lines of code, but with limited personalization. The latter is a little bit more verbose, but allows more details.

With ploty express, you basically choose the type of chart and give as arguments the dataframe, the columns for the x and y axis.

Let us for example create a bar plot (documentation [here](https://plotly.com/python/bar-charts/)) to visualize the evolution of the annual energy consumption year to year.

Before doing this, let us create a `year` column in the `yearly_energy_df` to facilitate our job:

#### Exercise 1️⃣ - Create year column in `yearly_energy_df`

In [ ]:
yearly_energy_df["year"] = yearly_energy_df.index.year

#### Exercise 2️⃣ - Create a simple bar plot

Create a bar plot (documentation [here](https://plotly.com/python/bar-charts/)) to visualize the evolution of the annual energy consumption year to year.

In [ ]:
px.bar(yearly_energy_df, x = "year", y = "Consumption")

With a line of code, you have just created an interactive bar plot. That is amazing.

You can notice that the labels of the axis are the name of the respective columns.

#### Exercise 3️⃣ - Improve the bar plot

You could want to improve a little better this plot: maybe include a title, set a different height and width, change the labels of the axis, specially the y-axis to set the units and also change the hover template (check this example [here](https://plotly.com/python/hover-text-and-formatting/#modifying-the-hovertemplate-of-a-plotly-express-figure)).

All this is possible with some extra lines of code, which are not complicated:

In [ ]:
fig = px.bar(yearly_energy_df, x = "year", y = "Consumption",
             labels={
             "Consumption": "Energy consumption (TWh)"
             },
       title = "Annual electricity consumption in France (in TWh)",
       width=600,
       height = 600)
fig.update_traces(hovertemplate='Year: %{x} <br>Annual Consumption: %{y:.1f} TWh')
fig.show()

Compare your last plot with the official results [here](https://analysesetdonnees.rte-france.com/consommation/synthese). The results are pretty close. The differences come from the approximation of the equation transforming power into energy (it has been considered that the power is constant over each 30-minute interval between each sample.)

You can explore the seasonality of the electricity consumption by plotting the consumption as a function of the month for each different year. 

#### Exercise 4️⃣ - Create date and year column in `monthly_energy_df`

Before making your plot, create a month and year columns in `monthly_energy_df`to facilitate your work.

In the month column, it is better to have the name of the month written. You can do this by using `pd.DatetimeIndex.strftime`(documentation [here](https://pandas.pydata.org/docs/reference/api/pandas.PeriodIndex.strftime.html)).

In [ ]:
monthly_energy_df["month"] = monthly_energy_df.index.strftime("%B")
monthly_energy_df["year"] = monthly_energy_df.index.year

#### Exercise 5️⃣ - Create a plot for monthly consumption for each year

You can have different lines using the `color` parameters, such as in this example [here](https://plotly.com/python/line-charts/#line-plots-with-column-encoding-color)

In [ ]:
fig = px.line(
    monthly_energy_df,
    x = "month",
    y = "Consumption",
    color = "year",
    labels = {
        "Consumption": "Energy consumption (TWh)",
    },
    title = "Monthly electricity consumption from 2012 - 2022",
    height = 600
)
fig.update_layout(hovermode= "x unified")
fig.update_traces(hovertemplate='%{y:.1f} TWh')
fig.show()

This is a very interesting plot. First, you can note that electricity consumption is higher from November to February, the colder months. On the contrary, the consumption is lower from June to August, the summertime. You can also notice that for some months, some years differ significantly from the rest. That is called _outliers_. You can notice at least three of them in this plot:
* A peak of consumption in February 2012. At this time, there was a particularly intense cold wave in Europe. The press reported a peak of electricity consumption in France on 8 February 2012:
    * [La France bat un record de consommation électrique](https://www.lesechos.fr/2012/02/la-france-bat-un-record-de-consommation-electrique-351079) by Les Echos, in French
    * [Nouveau record pour la consommation d'électricité](https://www.lemonde.fr/economie/article/2012/02/08/nouveau-record-pour-la-consommation-d-electricite_1640650_3234.html) by Le Monde, in French
    * [Electricity consumption in France to hit new high as cold snap continues](https://www.rfi.fr/en/france/20120207-electricity-consumption-france-set-new-record) by RFI
    * [Early 2012 European cold wave](https://en.wikipedia.org/wiki/Early_2012_European_cold_wave), article by Wikipedia
* A drop in electricity from March to June, with a notable drop in April (-15% compared to 2018, the second least consumption in this month). This coincides with the peak of Covid-19 crises. France was in lockdown from 17 March to 11 May 2020. Some articles stating those drops:
    * [Bilan Electrique 2020](https://assets.rte-france.com/prod/public/2021-03/Bilan%20electrique%202020_0.pdf) by RTE, in French
    * [Covid-19: infected electricity markets](https://www.tse-fr.eu/covid-19-infected-electricity-markets), by Stefan AMBEC and Claude CRAMPES in Toulouse School of Economics website.
* A drop in electricity in 2022 starting in October. There was a major energy crisis in France in 2022, due to Russia's invasion of Ukraine and drop in nuclear production, as you have already noticed. At the same time, government reinforced energy-saving messages, which seemed to be successful. Get more details on this drop in the [Annual Electricity Review 2022](https://analysesetdonnees.rte-france.com/en/electricity-review-keyfindings) by RTE.

You can go on and discover Plotly a little bit more, specially using graph objects, which have not been discussed in this notebook.

Let us get going and discover and new library.

### 📉 Forecast with Statsforecast

Statsforecast is a powerful library to make forecast for timeseries, which is data indexed to a timestamp, as are our dataframes.

This is not a Data Science notebook, so we will not get into many details on forecasting and modeling. Just enough so you can carry on and make a quick first forecast yourself.

In the next exercises, you will train a model to forecast electricity consumption for 2022, using the historical data from 2012 to 2021. You will do that using statsforecast. You can get inspired by this minimal example [here](https://nixtlaverse.nixtla.io/statsforecast/docs/getting-started/getting_started_short.html).

First, you need to put the data in `monthly_energy_df` in the way expected by statsforecast.

#### Exercise 6️⃣ - Get the data in a long format

Statsforecast expects data in a long format, such as in this example:

|    | unique_id     | ds         |   y |
|---:|:--------------|:-----------|----:|
|  0 | AirPassengers | 1949-01-01 | 112 |
|  1 | AirPassengers | 1949-02-01 | 118 |
|  2 | AirPassengers | 1949-03-01 | 132 |
|  3 | AirPassengers | 1949-04-01 | 129 |
|  4 | AirPassengers | 1949-05-01 | 121 |

You need to do the same with `monthly_energy_df`. You can do that with the following methods: `pd.DataFrame.stack` (documentation [here](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.stack.html)), `pd.Series.to_frame` (documentation [here](https://pandas.pydata.org/docs/reference/api/pandas.Series.to_frame.html)) and `pd.DataFrame.reset_index` (documentation [here](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.reset_index.html)). Remember to use only the column `Consumption`.

In [ ]:
monthly_long_format_df = monthly_energy_df[["Consumption"]].stack()
monthly_long_format_df = monthly_long_format_df.to_frame()
monthly_long_format_df = monthly_long_format_df.reset_index()
monthly_long_format_df.columns = ["ds", "unique_id", "y"]

Check if your `monthly_long_format_df` looks like this:

|    | ds         | unique_id   |       y |
|---:|:-----------|:------------|--------:|
|  0 | 2012-01-31 | Consumption | 50.7865 |
|  1 | 2012-02-29 | Consumption | 54.182  |
|  2 | 2012-03-31 | Consumption | 42.9455 |
|  3 | 2012-04-30 | Consumption | 39.9497 |
|  4 | 2012-05-31 | Consumption | 35.0388 |

In [ ]:
monthly_long_format_df.head()

#### Exercise 7️⃣ - Preparing train and test set

To evaluate the good fit of a model, you need to test it with data the model has not used to train. This is why you must divide the data in train and test set.

As stated before, you will train your model with historical data from 2012 to 2021 and use the data from 2022 to evaluate the performance of your model.

Consequently, your test set will consist of the last 12 points of data (the whole year 2022). You can use `pd.DataFrame.tail` (documentation [here](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.tail.html)). This means that for the train set, you must "let go" the 12 last points. You can use `pd.DataFrame.head` (documentation [here](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.head.html)), with a negative argument.

In [ ]:
train_set = monthly_long_format_df.head(n=-12)
test_set = monthly_long_format_df.tail(n=12)

#### Exercise 8️⃣ - Fit a AutoARIMA model

In a Data Science project, normally many models are tested and compared to each other. For the sake of this notebook, you are only going to use one model, which is called AutoARIMA. It combines auto-regressive, moving average and integration to make a forecast. You didn't get any of those? For now, no worries. The goal of this notebook is to get familiar with Python libraries and see what we can do with them. We do not want to become experts in AutoARIMA (or not yet 😀). Just know that this model can be fairly suitable for out type of data.

Let us train an AutoARIMA model with statsforecast.

First, you need to create a `StatsForecast` instance, which you have imported in the beginning of this notebook.

In this instance, you need to list the models you want to use. For our case, AutoARIMA. The model AutoARIMA accepts seasonality. For our case, there is a yearly seasonality for electricity consumption, so it needs to be set to 12 (1 year = 12 months). It also requests the frequency of the data, which is monthly (`ME` or "month-end)

Once the StatsForecast instance is created, you need to fit the AutoARIMA model on your train set. Check this quickstart [here](https://nixtlaverse.nixtla.io/statsforecast/docs/getting-started/getting_started_short.html) to get inspired.

In [ ]:
sf = StatsForecast(
    models=[AutoARIMA(season_length = 12)],
    freq='ME',
)
sf.fit(train_set)

#### Exercise 9️⃣ - Make a forecast

To make a forecast, you can use the `predict` method of StatsForecast instance, which take two arguments:
* `h`: the horizon of forecast. You would like to forecast for 12 months.
* `level`: calculates a prediction interval.  For example, level=[95] means that the model expects the real value to be inside that interval 95% of the times. Let us set it for this level.

In [ ]:
forecast_df = sf.predict(h=12, level=[95])

Check how `forecast_df` looks like:

In [ ]:
forecast_df.head()

Let us visualize the predictions:

In [ ]:
fig = sf.plot(test_set, forecast_df, level=[95], engine="plotly")
fig.update_layout(hovermode= "x unified", template="plotly_dark", height = 600)
fig.data[2]["hoverinfo"]="skip"
fig.data[2]["hovertemplate"]="None"
fig.show()

The predictions are out of the 95% interval confidence for October and November. Indeed, there was a major energy crisis in France in 2022 for these two months, as we have discussed in Exercise 5 of part 2.

This is some interesting to mention: this major energy crisis has never taken place before. So, even having historical data, it is very difficult to predict it.

But hey: for a very first forecast, without any optimization and no extra data to help, such as temperature or geo-political info, it looks kind of good, no? 

Maybe RTE will be a little more exigent, but you can be satisfied: you have forecasted quite well the electricity consumption in France in 2022.

Let's go for a very last exercise: quantify how good our prediction was with a metric. Let us check out the MAPE.

#### Exercise 🔟 - Calculate the MAPE of the predictions

The MAPE is the Mean Absolute Percentage Error. If we have a $Y$ actual values and $\hat{Y}$ forecasted values, the MAPE is calculated as follows:

$$ MAPE = \frac{1}{n}\sum_{i=1}^n \left\lvert{\frac{Y - \hat{Y}}{Y}}  \right\rvert $$

in which $n$ is the length of the vectors $Y$ and $\hat{Y}$. Let us calculate step by step:

In [ ]:
Y = test_set["y"].values
Y_hat = forecast_df["AutoARIMA"].values
pc_error = (Y - Y_hat)/(Y)
abs_pc_error = np.abs(pc_error)
MAPE = np.mean(abs_pc_error)
print(MAPE)

Our MAPE is 4,29%. That can be a little too much for RTE, but once again, this is an error from a model which has not been optimized. Furthermore, as we have seen in the beginning, RTE is interested in a more regular forecast, with shorter period, not a monthly one.

An important point to note for the MAPE is that as it is an average error, it does not capture variation in the errors. You can easily notice that the error for July is very low (0,34%), but the error in October for very high (14,6%).

## 🎉 Congratulations

You have just finished this notebook. Wow! That was a great job. You have:
* manipulated real data
* discovered new libraries and
* made a forecast for electricity consumption.

This is a great step for your journey to data 🥳

We hope you have enjoyed and that you will keep learning.